# Dynamic Time Warping: Elastic Alignment of Time Series

**Dynamic Time Warping** (DTW) is a similarity measure for time series that allows non-linear alignment: one series can be stretched or compressed locally to match the other. This makes DTW far more robust than the $L^2$ distance when two signals share the same shape but differ in timing or speed.

## Formal definition

Given two sequences $x = (x_1,\ldots,x_n)$ and $y = (y_1,\ldots,y_m)$, define the **cost matrix**
$$
C_{ij} = d(x_i, y_j),
$$
where $d$ is a local distance (e.g. squared Euclidean). A **warping path** is a sequence
$$
\pi = \bigl((i_1,j_1),\ldots,(i_K,j_K)\bigr) \subset \{1,\ldots,n\}\times\{1,\ldots,m\}
$$
satisfying the **boundary**, **monotonicity**, and **step-size** conditions. The DTW distance is
$$
\mathrm{DTW}(x,y) = \min_{\pi} \sum_{k=1}^K C_{i_k,j_k},
$$
computed efficiently via the recurrence
$$
D_{ij} = C_{ij} + \min\bigl(D_{i-1,j},\; D_{i,j-1},\; D_{i-1,j-1}\bigr),
$$
with $D_{11} = C_{11}$ and the optimal path recovered by backtracking.

**Key properties:**
- $\mathrm{DTW}(x,y) = 0$ iff $x = y$ (up to resampling).
- Satisfies the triangle inequality *only* with the Sakoe–Chiba band constraint.
- Complexity: $O(nm)$ time and space (reducible with pruning).

DTW is widely used in speech recognition, gesture classification, time-series clustering, financial signal comparison, and medical waveform analysis.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

plt.rcParams["figure.dpi"] = 120

## DTW implementation

We implement the standard $O(nm)$ DP algorithm. The cumulative cost matrix $D$ is filled row by row; the optimal warping path is recovered by greedy backtracking from $(n,m)$ to $(1,1)$.

In [ ]:
def dtw_distance(x, y, dist="square"):
    """Compute DTW distance and optimal warping path.
    Returns (distance, path) where path is a list of (i,j) index pairs."""
    n, m = len(x), len(y)
    # cost matrix
    if dist == "square":
        C = (x[:, None] - y[None, :]) ** 2
    else:
        C = np.abs(x[:, None] - y[None, :])

    # accumulate
    D = np.full((n, m), np.inf)
    D[0, 0] = C[0, 0]
    for i in range(1, n):
        D[i, 0] = D[i-1, 0] + C[i, 0]
    for j in range(1, m):
        D[0, j] = D[0, j-1] + C[0, j]
    for i in range(1, n):
        for j in range(1, m):
            D[i, j] = C[i, j] + min(D[i-1, j], D[i, j-1], D[i-1, j-1])

    # backtrack
    path = [(n-1, m-1)]
    i, j = n-1, m-1
    while i > 0 or j > 0:
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            best = np.argmin([D[i-1, j-1], D[i-1, j], D[i, j-1]])
            if best == 0:
                i -= 1; j -= 1
            elif best == 1:
                i -= 1
            else:
                j -= 1
        path.append((i, j))
    path.reverse()
    return D[-1, -1], np.array(path), C, D


print("DTW ready.")

## Warped sine: DTW vs Euclidean

Consider two sinusoids at the same frequency, where one has a **non-linear time warp** $\psi(t) = \tfrac{1}{2} + \tfrac{1}{2}\,\mathrm{sign}(2t-1)|2t-1|^3$. The Euclidean ($L^2$) distance is large because the signals are out of phase. DTW finds the optimal temporal alignment and recovers a near-zero cost.

In [ ]:
n = 300
t = np.linspace(0, 1, n)
f = 8

# non-linear time warp
psi = 0.5 + 0.5 * np.sign(2*t - 1) * np.abs(2*t - 1)**3
x = np.cos(2 * np.pi * f * psi)
y = np.cos(2 * np.pi * f * t)

dtw_dist, path, C, D = dtw_distance(x, y)
l2_dist = np.sum((x - y)**2)

print(f"L2 distance²  = {l2_dist:.1f}")
print(f"DTW distance² = {dtw_dist:.3f}")

fig = plt.figure(figsize=(12, 7))
gs = gridspec.GridSpec(2, 2, width_ratios=[3, 1], height_ratios=[1, 3],
                       hspace=0.05, wspace=0.05)

# signals
ax_x = fig.add_subplot(gs[0, 0])
ax_x.plot(t, x, 'b', lw=1.8, label="$x$ (warped)")
ax_x.set_xlim(0, 1); ax_x.set_xticks([])
ax_x.set_ylabel("$x$"); ax_x.legend(loc="upper right", fontsize=8)

ax_y = fig.add_subplot(gs[1, 1])
ax_y.plot(y, t, 'r', lw=1.8, label="$y$ (uniform)")
ax_y.set_ylim(0, 1); ax_y.set_yticks([])
ax_y.set_xlabel("$y$"); ax_y.legend(loc="upper right", fontsize=8)

# cost + path
ax_c = fig.add_subplot(gs[1, 0])
ax_c.imshow(C.T, origin="lower", aspect="auto", cmap="hot_r",
            extent=[0, 1, 0, 1])
pi = path / (n - 1)
ax_c.plot(pi[:, 0], pi[:, 1], 'c-', lw=2, label="warping path")
ax_c.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5, label="diagonal (Euclid)")
ax_c.set_xlabel("$i/n$  (index in $x$)")
ax_c.set_ylabel("$j/m$  (index in $y$)")
ax_c.legend(fontsize=8)

fig.suptitle(f"DTW warped sine  |  L²={l2_dist:.0f}  |  DTW²={dtw_dist:.2f}",
             y=1.01)
plt.show()

## Alignment visualization

The warping path defines a **correspondence** between the two time series. We draw selected matched pairs as vertical connectors. Notice that multiple points in $x$ may map to a single point in $y$ (or vice versa), allowing local stretching.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
gap = 2.5
ax.plot(t, x - gap, 'b', lw=2, label="$x$ (warped)")
ax.plot(t, y + gap, 'r', lw=2, label="$y$ (uniform)")

# draw correspondences at regular intervals along the path
skip = max(1, len(path) // 50)
for idx in range(0, len(path), skip):
    i, j = path[idx]
    alpha = 0.3
    ax.plot([t[i], t[j]], [x[i] - gap, y[j] + gap], 'k-', lw=0.7, alpha=alpha)

ax.set_xlim(0, 1)
ax.set_xlabel("time")
ax.legend()
ax.set_title("DTW alignment: correspondences between the two time series")
ax.axis("off")
plt.tight_layout()
plt.show()

## Sakoe–Chiba band constraint

Unrestricted DTW can produce degenerate paths (extreme compressions). The **Sakoe–Chiba band** restricts the warping to paths within a diagonal band of half-width $w$:
$$
D_{ij} = \begin{cases} C_{ij} + \min(\ldots) & \text{if } |i - j| \leq w, \\ +\infty & \text{otherwise.} \end{cases}
$$
Larger $w$ gives more flexibility but $w = 0$ reduces DTW to Euclidean matching.

In [ ]:
def dtw_banded(x, y, w):
    n, m = len(x), len(y)
    C = (x[:, None] - y[None, :]) ** 2
    D = np.full((n, m), np.inf)
    for i in range(n):
        for j in range(m):
            if abs(i - j) > w:
                continue
            prev = [D[i-1, j-1] if i>0 and j>0 else np.inf,
                    D[i-1, j]   if i>0 else np.inf,
                    D[i, j-1]   if j>0 else np.inf]
            D[i, j] = C[i, j] + (0 if i==0 and j==0 else min(prev))
    return D[-1, -1], D


bands = [5, 20, 50, n-1]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, w in zip(axes, bands):
    dist_w, D_w = dtw_banded(x, y, w)
    mask = np.isfinite(D_w)
    ax.imshow(C.T, origin="lower", aspect="auto", cmap="hot_r")
    # show band
    band_mask = np.zeros_like(C)
    for i in range(n):
        for j in range(n):
            if abs(i-j) <= w:
                band_mask[i, j] = 1
    ax.contour(band_mask.T, levels=[0.5], colors='cyan', linewidths=1.5)
    ax.set_title(f"$w={w}$  DTW²={dist_w:.1f}", fontsize=9)
    ax.axis("off")
fig.suptitle("Sakoe–Chiba band: effect on cost matrix and path flexibility", y=1.02)
plt.tight_layout()
plt.show()

## Interactive: compare two custom signals

Generate pairs of signals with controlled phase shift or frequency difference, and compare their L² and DTW distances.

In [ ]:
def show_dtw(signal="warped-sine", freq=8, warp_power=3.0):
    n_s = 250
    t_s = np.linspace(0, 1, n_s)
    if signal == "warped-sine":
        psi_s = 0.5 + 0.5*np.sign(2*t_s-1)*np.abs(2*t_s-1)**warp_power
        xs = np.cos(2*np.pi*freq*psi_s)
        ys = np.cos(2*np.pi*freq*t_s)
    elif signal == "chirp-vs-sine":
        xs = np.cos(2*np.pi*(1 + freq*t_s)*t_s)
        ys = np.cos(2*np.pi*freq*t_s)
    else:  # phase shift
        phi = warp_power * np.pi / 4
        xs = np.cos(2*np.pi*freq*t_s + phi)
        ys = np.cos(2*np.pi*freq*t_s)

    d_dtw, path_s, C_s, _ = dtw_distance(xs, ys)
    d_l2 = np.sum((xs - ys)**2)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(t_s, xs, 'b', lw=2, label="$x$")
    axes[0].plot(t_s, ys, 'r', lw=2, label="$y$")
    axes[0].set_title(f"Signals  |  L²={d_l2:.1f}  |  DTW²={d_dtw:.2f}")
    axes[0].legend(); axes[0].set_xlabel("$t$")

    axes[1].imshow(C_s.T, origin="lower", aspect="auto", cmap="hot_r")
    pi_s = path_s
    axes[1].plot(pi_s[:,0], pi_s[:,1], 'c-', lw=2)
    axes[1].plot([0, n_s-1], [0, n_s-1], 'w--', lw=1, alpha=0.5)
    axes[1].set_title("Cost matrix + warping path")
    axes[1].set_xlabel("$i$"); axes[1].set_ylabel("$j$")
    plt.tight_layout(); plt.show()

interact(
    show_dtw,
    signal=Dropdown(options=["warped-sine","chirp-vs-sine","phase-shift"],
                    description="signal"),
    freq=IntSlider(value=8, min=2, max=20, step=1, description="freq"),
    warp_power=FloatSlider(value=3.0, min=1.0, max=6.0, step=0.5,
                           description="warp"),
);

## Bibliographical resources

- Sakoe, H. and Chiba, S. (1978). Dynamic programming algorithm optimization for spoken word recognition. *IEEE Transactions on Acoustics, Speech, and Signal Processing*, 26(1), 43–49.
- Berndt, D. J. and Clifford, J. (1994). Using dynamic time warping to find patterns in time series. *KDD Workshop*, 359–370.
- Müller, M. (2007). *Information Retrieval for Music and Motion*. Springer.
- Salvador, S. and Chan, P. (2007). Toward accurate dynamic time warping in linear time and space. *Intelligent Data Analysis*, 11(5), 561–580.
- Tavenard, R. et al. (2020). tslearn, a machine learning toolkit for time series data. *JMLR*, 21(118), 1–6.